# 🛒 Olist eCommerce Store Performance Audit
## Phase 1 — Data Loading & Cleaning

**Project by:** Smit Bhavsar  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Kaggle)  
**Tools:** Python, Pandas, NumPy  

---

## Business Problem

An eCommerce marketplace wants to understand:

- Why is customer retention low?
- Which product categories drive the most revenue?
- Where are delivery operations failing — and how does it affect customer satisfaction?

This notebook handles **data loading, exploration, and cleaning** — the foundation of the entire project.

---

## Step 1 — Install & Import Libraries

In [19]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)   # show all columns
pd.set_option('display.max_rows', 100)        # show up to 50 rows
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places

# Chart style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

print('✅ All libraries imported successfully!')

✅ All libraries imported successfully!


## Step 2 — Set Data Path

In [20]:
# Define data path
DATA_PATH = '../data/'   # assumes CSVs are in a 'data' subfolder

# List all CSV files in the folder to verify
files = [f for f in os.listdir(DATA_PATH) if f.endswith('.csv')]
print(f'📂 Found {len(files)} CSV files:')
for f in sorted(files):
    print(f'   → {f}')

📂 Found 9 CSV files:
   → olist_customers_dataset.csv
   → olist_geolocation_dataset.csv
   → olist_order_items_dataset.csv
   → olist_order_payments_dataset.csv
   → olist_order_reviews_dataset.csv
   → olist_orders_dataset.csv
   → olist_products_dataset.csv
   → olist_sellers_dataset.csv
   → product_category_name_translation.csv


## Step 3 — Load All 9 CSV Files

In [6]:
# Load each CSV into its own dataframe
# We give short, clear variable names for easy use later

df_orders       = pd.read_csv(DATA_PATH + 'olist_orders_dataset.csv')
df_order_items  = pd.read_csv(DATA_PATH + 'olist_order_items_dataset.csv')
df_customers    = pd.read_csv(DATA_PATH + 'olist_customers_dataset.csv')
df_payments     = pd.read_csv(DATA_PATH + 'olist_order_payments_dataset.csv')
df_reviews      = pd.read_csv(DATA_PATH + 'olist_order_reviews_dataset.csv')
df_products     = pd.read_csv(DATA_PATH + 'olist_products_dataset.csv')
df_sellers      = pd.read_csv(DATA_PATH + 'olist_sellers_dataset.csv')
df_geo          = pd.read_csv(DATA_PATH + 'olist_geolocation_dataset.csv')
df_translation  = pd.read_csv(DATA_PATH + 'product_category_name_translation.csv')

print('✅ All 9 datasets loaded!')
print()

✅ All 9 datasets loaded!



In [12]:
# Quick size check for each table
datasets = {
    'Orders':        df_orders,
    'Order Items':   df_order_items,
    'Customers':     df_customers,
    'Payments':      df_payments,
    'Reviews':       df_reviews,
    'Products':      df_products,
    'Sellers':       df_sellers,
    'Geolocation':   df_geo,
    'Translation':   df_translation
}

print(f'{'Dataset':<20} {'Rows':>10} {'Columns':>10}')
for name, df in datasets.items():
    print(f'{name:<20} {df.shape[0]:>10,} {df.shape[1]:>10}')

Dataset                    Rows    Columns
Orders                   99,441          8
Order Items             112,650          7
Customers                99,441          5
Payments                103,886          5
Reviews                  99,224          7
Products                 32,951          9
Sellers                   3,095          4
Geolocation           1,000,163          5
Translation                  71          2


## Step 4 — Explore Each Table

In [18]:
# ── ORDERS TABLE ── The core table. Everything links back to order_id.
print('=== ORDERS TABLE ===')
print(df_orders.head())

=== ORDERS TABLE ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26 14:31

In [19]:
print('Columns:', df_orders.columns.tolist())

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [20]:
print('Data types:\n', df_orders.dtypes)

Data types:
 order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [23]:
# ── ORDER ITEMS TABLE ── Has price, freight, product info per order
print('=== ORDERS ITEMS TABLE ===')
print(df_orders.head())

=== ORDERS ITEMS TABLE ===
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_status order_purchase_timestamp    order_approved_at  \
0    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15   
1    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27   
2    delivered      2018-08-08 08:38:49  2018-08-08 08:55:23   
3    delivered      2017-11-18 19:28:06  2017-11-18 19:45:59   
4    delivered      2018-02-13 21:18:39  2018-02-13 22:20:29   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2018-07-26

In [24]:
print('Unique orders in items table:', df_order_items['order_id'].nunique())

Unique orders in items table: 98666


In [25]:
# ── CUSTOMERS TABLE ──
# IMPORTANT: customer_id changes per order. customer_unique_id is the TRUE repeat buyer identifier.
print('=== CUSTOMERS TABLE ===')
print(df_customers.head())

=== CUSTOMERS TABLE ===
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  


In [26]:
print('Total customer_id rows:       ', df_customers['customer_id'].nunique())

Total customer_id rows:        99441


In [27]:
print('Total unique customers (real):', df_customers['customer_unique_id'].nunique())

Total unique customers (real): 96096


In [28]:
# ── PAYMENTS TABLE ──
print('=== PAYMENTS TABLE ===')
print(df_payments.head())

=== PAYMENTS TABLE ===
                           order_id  payment_sequential payment_type  \
0  b81ef226f3fe1789b1e8b2acac839d17                   1  credit_card   
1  a9810da82917af2d9aefd1278f1dcfa0                   1  credit_card   
2  25e8ea4e93396b6fa0d3dd708e76c1bd                   1  credit_card   
3  ba78997921bbcdc1373bb41e913ab953                   1  credit_card   
4  42fdf880ba16b47b59251dd489d4441a                   1  credit_card   

   payment_installments  payment_value  
0                     8          99.33  
1                     1          24.39  
2                     1          65.71  
3                     8         107.78  
4                     2         128.45  


In [29]:
print('Payment types:', df_payments['payment_type'].unique())

Payment types: <StringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str


In [30]:
# ── REVIEWS TABLE ──
print('=== REVIEWS TABLE ===')
print(df_reviews.head())

=== REVIEWS TABLE ===
                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                                                NaN  2018-01-18 00:00:00   
1                                                NaN  2018-03-10 00:00:00   
2                                                NaN  2018-02-17 00

In [31]:
print('Review score distribution:')
print(df_reviews['review_score'].value_counts().sort_index())

Review score distribution:
review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64


In [32]:
# ── PRODUCTS TABLE ──
print('=== PRODUCTS TABLE ===')
print(df_products.head())

=== PRODUCTS TABLE ===
                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perfumaria   
1  3aa071139cb16b67ca9e5dea641aaa2f                  artes   
2  96bd76ec8810374ed1b65e291975717f          esporte_lazer   
3  cef67bcfe19066a932b7673e239eb23d                  bebes   
4  9dc1a7de274444849c219cff195d0b71  utilidades_domesticas   

   product_name_lenght  product_description_lenght  product_photos_qty  \
0                40.00                      287.00                1.00   
1                44.00                      276.00                1.00   
2                46.00                      250.00                1.00   
3                27.00                      261.00                1.00   
4                37.00                      402.00                4.00   

   product_weight_g  product_length_cm  product_height_cm  product_width_cm  
0            225.00              16.00              10.00             14.00  
1  

In [33]:
print('Total unique categories:', df_products['product_category_name'].nunique())

Total unique categories: 73


## Step 5 — Check Missing Values in Every Table

In [36]:
print('=== MISSING VALUE REPORT ===')
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]  # only show columns that have nulls
    if len(missing) > 0:
        print(f'── {name} ──')
        for col, count in missing.items():
            pct = (count / len(df)) * 100
            print(f'   {col}: {count:,} missing ({pct:.1f}%)')
        print()
    else:
        print(f'── {name}: ✅ No missing values')

=== MISSING VALUE REPORT ===
── Orders ──
   order_approved_at: 160 missing (0.2%)
   order_delivered_carrier_date: 1,783 missing (1.8%)
   order_delivered_customer_date: 2,965 missing (3.0%)

── Order Items: ✅ No missing values
── Customers: ✅ No missing values
── Payments: ✅ No missing values
── Reviews ──
   review_comment_title: 87,656 missing (88.3%)
   review_comment_message: 58,247 missing (58.7%)

── Products ──
   product_category_name: 610 missing (1.9%)
   product_name_lenght: 610 missing (1.9%)
   product_description_lenght: 610 missing (1.9%)
   product_photos_qty: 610 missing (1.9%)
   product_weight_g: 2 missing (0.0%)
   product_length_cm: 2 missing (0.0%)
   product_height_cm: 2 missing (0.0%)
   product_width_cm: 2 missing (0.0%)

── Sellers: ✅ No missing values
── Geolocation: ✅ No missing values
── Translation: ✅ No missing values


## Step 6 — Clean the Orders Table
This is the most important cleaning step — fix date columns and handle missing delivery dates.

In [41]:
# Convert all date columns from string → datetime
# This is essential for any time-based analysis
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols :
    df_orders[col] = pd.to_datetime(df_orders[col])

print('✅ Date columns converted to datetime')
print('Date column types:')
print(df_orders[date_cols].dtypes)

✅ Date columns converted to datetime
Date column types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [44]:
# Add useful time-based columns for analysis

df_orders['order_year']   = df_orders['order_purchase_timestamp'].dt.year
df_orders['order_month']  = df_orders['order_purchase_timestamp'].dt.month
df_orders['order_month_name'] = df_orders['order_purchase_timestamp'].dt.strftime('%b %Y')
df_orders['order_day_of_week'] = df_orders['order_purchase_timestamp'].dt.day_name()
df_orders['order_hour']   = df_orders['order_purchase_timestamp'].dt.hour

# Calculate delivery delay in days
# Positive = delivered AFTER estimated date (late)
# Negative = delivered BEFORE estimated date (early)
df_orders['delivery_delay_days'] = (
    df_orders['order_delivered_customer_date'] - 
    df_orders['order_estimated_delivery_date']
).dt.days

# Calculate actual delivery time from purchase to delivery
df_orders['actual_delivery_days'] = (
    df_orders['order_delivered_customer_date'] - 
    df_orders['order_purchase_timestamp']
).dt.days

# Flag late orders (delay > 0 means delivered after promised date)
df_orders['is_late'] = df_orders['delivery_delay_days'] > 0

print('✅ New columns added to orders table:')
new_cols = ['order_year','order_month','order_month_name','order_day_of_week',
            'order_hour','delivery_delay_days','actual_delivery_days','is_late']
print(df_orders[new_cols].head(5))

✅ New columns added to orders table:
   order_year  order_month order_month_name order_day_of_week  order_hour  \
0        2017           10         Oct 2017            Monday          10   
1        2018            7         Jul 2018           Tuesday          20   
2        2018            8         Aug 2018         Wednesday           8   
3        2017           11         Nov 2017          Saturday          19   
4        2018            2         Feb 2018           Tuesday          21   

   delivery_delay_days  actual_delivery_days  is_late  
0                -8.00                  8.00    False  
1                -6.00                 13.00    False  
2               -18.00                  9.00    False  
3               -13.00                 13.00    False  
4               -10.00                  2.00    False  


## Step 7 — Clean the Products Table
Translate Portuguese category names to English.

In [45]:
# Merge English category names into products table
df_products = df_products.merge(
    df_translation,
    on='product_category_name',
    how='left'
)

# Fill any categories that didn't get a translation with the Portuguese name
df_products['product_category_name_english'] = (
    df_products['product_category_name_english']
    .fillna(df_products['product_category_name'])
)

print('✅ English category names added')
print()
print('Sample categories (English):')
print(df_products['product_category_name_english'].value_counts().head(10))

✅ English category names added

Sample categories (English):
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


## Step 8 — Clean the Reviews Table


In [46]:
# Convert review dates
df_reviews['review_creation_date']    = pd.to_datetime(df_reviews['review_creation_date'])
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])

# Keep only one review per order (take the latest one if duplicates exist)
df_reviews = (
    df_reviews
    .sort_values('review_creation_date', ascending=False)
    .drop_duplicates(subset='order_id', keep='first')
)

print('✅ Reviews table cleaned')
print(f'Total reviews after dedup: {len(df_reviews):,}')

✅ Reviews table cleaned
Total reviews after dedup: 98,673


## Step 9 — Build the Master Orders DataFrame
This is the most important step. We join all tables into one master dataframe that we use for all analysis.

In [48]:
# Step 9a: Aggregate order_items per order
# (one order can have multiple items, we sum price and freight)
order_items_agg = df_order_items.groupby('order_id').agg(
    total_items      = ('order_item_id', 'count'),       # number of items in order
    total_price      = ('price', 'sum'),                  # total product price
    total_freight    = ('freight_value', 'sum'),          # total shipping cost
    total_revenue    = ('price', lambda x: x.sum() + df_order_items.loc[x.index, 'freight_value'].sum()),
).reset_index()

# Recalculate total_revenue cleanly
order_items_agg['total_revenue'] = order_items_agg['total_price'] + order_items_agg['total_freight']

print('✅ Order items aggregated')
print(order_items_agg.head(3))

✅ Order items aggregated
                           order_id  total_items  total_price  total_freight  \
0  00010242fe8c5a6d1ba2dd792cb16214            1        58.90          13.29   
1  00018f77f2f0320c557190d7a144bdd3            1       239.90          19.93   
2  000229ec398224ef6ca0657da4fc703e            1       199.00          17.87   

   total_revenue  
0          72.19  
1         259.83  
2         216.87  


In [49]:
# Step 9b: Get main product category per order
# Join order_items → products to get category, then take first category per order
order_category = (
    df_order_items[['order_id','product_id']]
    .merge(df_products[['product_id','product_category_name_english']], on='product_id', how='left')
    .groupby('order_id')['product_category_name_english']
    .first()
    .reset_index()
    .rename(columns={'product_category_name_english': 'product_category'})
)

print('✅ Product category per order extracted')
print(order_category.head(3))

✅ Product category per order extracted
                           order_id product_category
0  00010242fe8c5a6d1ba2dd792cb16214       cool_stuff
1  00018f77f2f0320c557190d7a144bdd3         pet_shop
2  000229ec398224ef6ca0657da4fc703e  furniture_decor


In [50]:
# Step 9c: Aggregate payments per order
payments_agg = df_payments.groupby('order_id').agg(
    payment_type          = ('payment_type', 'first'),
    payment_installments  = ('payment_installments', 'max'),
    payment_value         = ('payment_value', 'sum')
).reset_index()

print('✅ Payments aggregated')
print(payments_agg.head(3))

✅ Payments aggregated
                           order_id payment_type  payment_installments  \
0  00010242fe8c5a6d1ba2dd792cb16214  credit_card                     2   
1  00018f77f2f0320c557190d7a144bdd3  credit_card                     3   
2  000229ec398224ef6ca0657da4fc703e  credit_card                     5   

   payment_value  
0          72.19  
1         259.83  
2         216.87  


In [51]:
# Step 9d: Build the MASTER dataframe — join everything together
df_master = (
    df_orders
    .merge(df_customers[['customer_id','customer_unique_id','customer_city','customer_state']],
           on='customer_id', how='left')
    .merge(order_items_agg, on='order_id', how='left')
    .merge(order_category,  on='order_id', how='left')
    .merge(payments_agg,    on='order_id', how='left')
    .merge(df_reviews[['order_id','review_score']], on='order_id', how='left')
)

print('✅ MASTER dataframe created!')
print(f'Shape: {df_master.shape}')
print()
print('Columns in master dataframe:')
print(df_master.columns.tolist())

✅ MASTER dataframe created!
Shape: (99441, 28)

Columns in master dataframe:
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'order_year', 'order_month', 'order_month_name', 'order_day_of_week', 'order_hour', 'delivery_delay_days', 'actual_delivery_days', 'is_late', 'customer_unique_id', 'customer_city', 'customer_state', 'total_items', 'total_price', 'total_freight', 'total_revenue', 'product_category', 'payment_type', 'payment_installments', 'payment_value', 'review_score']


In [53]:
# Step 9e: Filter to only DELIVERED orders for revenue analysis
# (cancelled/unavailable orders should not count as revenue)

print('Order status breakdown:')
print(df_master['order_status'].value_counts())

# We keep all orders in df_master but create a delivered-only view
df_delivered = df_master[df_master['order_status'] == 'delivered'].copy()

print(f'Total orders:     {len(df_master):,}')
print(f'Delivered orders: {len(df_delivered):,}')
print(f'Delivered rate:   {len(df_delivered)/len(df_master)*100:.1f}%')

Order status breakdown:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
Total orders:     99,441
Delivered orders: 96,478
Delivered rate:   97.0%


## Step 10 — Final Data Quality Check

In [64]:
print('=== MASTER DATAFRAME SUMMARY ===')
print()
print(f'Total orders:              {len(df_master):>10,}')
print(f'Unique customers:          {df_master["customer_unique_id"].nunique():>10,}')
print(f'Date range:                {df_master["order_purchase_timestamp"].min().date()} to {df_master["order_purchase_timestamp"].max().date()}')
print(f'Total revenue (delivered): R$ {df_delivered["total_revenue"].sum():>10,.2f}')
print(f'Avg order value (AOV):     R$ {df_delivered["total_revenue"].mean():>10,.2f}')
print(f'Avg review score:          {df_delivered["review_score"].mean():>10.2f} / 5')
print(f'Avg delivery delay:        {df_delivered["delivery_delay_days"].mean():>10.1f} days')
print(f'Late delivery rate:        {df_delivered["is_late"].mean()*100:>10.1f}%')
print(f'Product categories:        {df_master["product_category"].nunique():>10}')

=== MASTER DATAFRAME SUMMARY ===

Total orders:                  99,441
Unique customers:              96,096
Date range:                2016-09-04 to 2018-10-17
Total revenue (delivered): R$ 15,419,773.75
Avg order value (AOV):     R$     159.83
Avg review score:                4.16 / 5
Avg delivery delay:             -11.9 days
Late delivery rate:               6.8%
Product categories:                73


In [30]:
# Fix 1 — Fill missing product categories
df_master['product_category'] = df_master['product_category'].fillna('unknown_category')
df_delivered['product_category'] = df_delivered['product_category'].fillna('unknown_category')

# Fix 2 — Fill missing review scores with median (3.0)
df_master['review_score'] = df_master['review_score'].fillna(df_master['review_score'].median())
df_delivered['review_score'] = df_delivered['review_score'].fillna(df_delivered['review_score'].median())

# Fix 3 — Fill missing delivery dates with median delivery days
median_delay = df_delivered['delivery_delay_days'].median()
median_actual = df_delivered['actual_delivery_days'].median()
df_master['delivery_delay_days'] = df_master['delivery_delay_days'].fillna(median_delay)
df_delivered['delivery_delay_days'] = df_delivered['delivery_delay_days'].fillna(median_delay)
df_master['actual_delivery_days'] = df_master['actual_delivery_days'].fillna(median_actual)
df_delivered['actual_delivery_days'] = df_delivered['actual_delivery_days'].fillna(median_actual)

print("✅ Nulls handled!")
print(f"product_category nulls: {df_delivered['product_category'].isnull().sum()}")
print(f"review_score nulls:     {df_delivered['review_score'].isnull().sum()}")
print(f"delivery_delay nulls:   {df_delivered['delivery_delay_days'].isnull().sum()}")

✅ Nulls handled!
product_category nulls: 0
review_score nulls:     0
delivery_delay nulls:   0


## Step 11 — Save Cleaned Data for Next Phases

In [31]:
# Save the master dataframe so we can load it directly in Phase 2 and 3
import os
os.makedirs('cleaned_data', exist_ok=True)

df_master.to_csv('cleaned_data/olist_master.csv', index=False)
df_delivered.to_csv('cleaned_data/olist_delivered.csv', index=False)
df_products.to_csv('cleaned_data/olist_products_clean.csv', index=False)
df_sellers.to_csv('cleaned_data/olist_sellers_clean.csv', index=False)

print('✅ Cleaned data saved to /cleaned_data folder!')
print()
print('Files saved:')
for f in os.listdir('cleaned_data'):
    size = os.path.getsize(f'cleaned_data/{f}') / 1024
    print(f'   → {f} ({size:.0f} KB)')

✅ Cleaned data saved to /cleaned_data folder!

Files saved:
   → olist_delivered.csv (30369 KB)
   → olist_master.csv (31199 KB)
   → olist_products_clean.csv (3204 KB)
   → olist_sellers_clean.csv (163 KB)


In [33]:
# ── NULL VALUE CHECK — olist_master ───────────────────────
print('=' * 50)
print('NULL CHECK — olist_master')
print('=' * 50)
master_nulls = df_master.isnull().sum()
master_nulls = master_nulls[master_nulls > 0]
if len(master_nulls) == 0:
    print('✅ No nulls found!')
else:
    for col, count in master_nulls.items():
        pct = count / len(df_master) * 100
        print(f'   ⚠️  {col}: {count:,} nulls ({pct:.2f}%)')


NULL CHECK — olist_master
   ⚠️  order_approved_at: 160 nulls (0.16%)
   ⚠️  order_delivered_carrier_date: 1,783 nulls (1.79%)
   ⚠️  order_delivered_customer_date: 2,965 nulls (2.98%)
   ⚠️  total_items: 775 nulls (0.78%)
   ⚠️  total_price: 775 nulls (0.78%)
   ⚠️  total_freight: 775 nulls (0.78%)
   ⚠️  total_revenue: 775 nulls (0.78%)
   ⚠️  payment_type: 1 nulls (0.00%)
   ⚠️  payment_installments: 1 nulls (0.00%)
   ⚠️  payment_value: 1 nulls (0.00%)


In [34]:
# ── ADDITIONAL NULL FIXES for olist_master ────────────────
print('=' * 50)
print('FIXING MASTER TABLE ADDITIONAL NULLS...')
print('=' * 50)

# Fix date columns — these are NULL because order was not
# delivered (cancelled/unavailable). Fill with a placeholder
# string so they are recognizable in analysis.
# We do NOT fill with a fake date — that would corrupt analysis.
# Instead we leave date columns as-is (they are correct nulls)
# and only fix the numeric/text columns below.

# Fix total_items, total_price, total_freight, total_revenue
# These 775 orders have no order_items — fill with 0
# because they are ghost orders with no products attached
df_master['total_items']   = df_master['total_items'].fillna(0)
df_master['total_price']   = df_master['total_price'].fillna(0)
df_master['total_freight'] = df_master['total_freight'].fillna(0)
df_master['total_revenue'] = df_master['total_revenue'].fillna(0)
print('✅ total_items / total_price / total_freight / total_revenue → filled with 0')
print('   (775 ghost orders with no products — correctly set to zero)')

# Fix payment columns — 1 order with no payment record
df_master['payment_type']         = df_master['payment_type'].fillna('unknown')
df_master['payment_installments'] = df_master['payment_installments'].fillna(0)
df_master['payment_value']        = df_master['payment_value'].fillna(0)
print('✅ payment_type / payment_installments / payment_value → filled')

# Date nulls — leave order_approved_at,
# order_delivered_carrier_date, order_delivered_customer_date
# as NULL. These are CORRECT nulls — they represent orders
# that were never approved or delivered. Filling them with
# fake dates would corrupt your delivery delay analysis.
print()
print('ℹ️  Date column nulls are LEFT AS NULL intentionally:')
print('   order_approved_at              → 160 nulls = payment failed orders')
print('   order_delivered_carrier_date   → 1,783 nulls = never shipped')
print('   order_delivered_customer_date  → 2,965 nulls = never delivered')
print('   These are CORRECT — do not fill date nulls with fake values')

print()

# Re-save master after fixes
df_master.to_csv('../cleaned_data/olist_master.csv', index=False)
print('✅ olist_master.csv re-saved with fixes applied!')

print()

# Final check — what nulls remain in master
print('=' * 50)
print('REMAINING NULLS in olist_master after fix:')
print('=' * 50)
remaining = df_master.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print('✅ Zero nulls remaining!')
else:
    for col, count in remaining.items():
        pct = count / len(df_master) * 100
        print(f'   {col}: {count:,} nulls ({pct:.2f}%) — intentional date nulls')

FIXING MASTER TABLE ADDITIONAL NULLS...
✅ total_items / total_price / total_freight / total_revenue → filled with 0
   (775 ghost orders with no products — correctly set to zero)
✅ payment_type / payment_installments / payment_value → filled

ℹ️  Date column nulls are LEFT AS NULL intentionally:
   order_approved_at              → 160 nulls = payment failed orders
   order_delivered_carrier_date   → 1,783 nulls = never shipped
   order_delivered_customer_date  → 2,965 nulls = never delivered
   These are CORRECT — do not fill date nulls with fake values

✅ olist_master.csv re-saved with fixes applied!

REMAINING NULLS in olist_master after fix:
   order_approved_at: 160 nulls (0.16%) — intentional date nulls
   order_delivered_carrier_date: 1,783 nulls (1.79%) — intentional date nulls
   order_delivered_customer_date: 2,965 nulls (2.98%) — intentional date nulls


In [35]:
df_master.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
order_year                          0
order_month                         0
order_month_name                    0
order_day_of_week                   0
order_hour                          0
delivery_delay_days                 0
actual_delivery_days                0
is_late                             0
customer_unique_id                  0
customer_city                       0
customer_state                      0
total_items                         0
total_price                         0
total_freight                       0
total_revenue                       0
product_category                    0
payment_type                        0
payment_installments                0
payment_valu

In [36]:
# ── NULL VALUE CHECK — olist_delivered ────────────────────
print('=' * 50)
print('NULL CHECK — olist_delivered')
print('=' * 50)
delivered_nulls = df_delivered.isnull().sum()
delivered_nulls = delivered_nulls[delivered_nulls > 0]
if len(delivered_nulls) == 0:
    print('✅ No nulls found!')
else:
    for col, count in delivered_nulls.items():
        pct = count / len(df_delivered) * 100
        print(f'   ⚠️  {col}: {count:,} nulls ({pct:.2f}%)')

NULL CHECK — olist_delivered
   ⚠️  order_approved_at: 14 nulls (0.01%)
   ⚠️  order_delivered_carrier_date: 2 nulls (0.00%)
   ⚠️  order_delivered_customer_date: 8 nulls (0.01%)
   ⚠️  payment_type: 1 nulls (0.00%)
   ⚠️  payment_installments: 1 nulls (0.00%)
   ⚠️  payment_value: 1 nulls (0.00%)


In [37]:
# ── FIX NULLS in olist_delivered ──────────────────────────
print('=' * 50)
print('FIXING olist_delivered NULLS...')
print('=' * 50)

# Date nulls — these 8+14+2 rows are edge cases where the
# order status shows "delivered" but timestamps were not
# recorded properly in Olist's system.
# Fix: fill with the column median so delivery calculations
# still work correctly for these few rows.

median_approved     = df_delivered['order_approved_at'].median()
median_carrier      = df_delivered['order_delivered_carrier_date'].median()
median_delivered    = df_delivered['order_delivered_customer_date'].median()

df_delivered['order_approved_at'] = (
    df_delivered['order_approved_at'].fillna(median_approved)
)
df_delivered['order_delivered_carrier_date'] = (
    df_delivered['order_delivered_carrier_date'].fillna(median_carrier)
)
df_delivered['order_delivered_customer_date'] = (
    df_delivered['order_delivered_customer_date'].fillna(median_delivered)
)
print('✅ order_approved_at          → 14 nulls filled with median date')
print('✅ order_delivered_carrier_date → 2 nulls filled with median date')
print('✅ order_delivered_customer_date → 8 nulls filled with median date')

# Payment nulls — 1 single order
df_delivered['payment_type']         = df_delivered['payment_type'].fillna('unknown')
df_delivered['payment_installments'] = df_delivered['payment_installments'].fillna(0)
df_delivered['payment_value']        = df_delivered['payment_value'].fillna(0)
print('✅ payment_type / installments / value → 1 null filled')

print()

# Re-save after fix
df_delivered.to_csv('../cleaned_data/olist_delivered.csv', index=False)
print('✅ olist_delivered.csv re-saved!')

print()

# ── FINAL VERIFICATION ────────────────────────────────────
print('=' * 50)
print('FINAL VERIFICATION — all 4 cleaned files')
print('=' * 50)

all_clean = True
for name, df in [
    ('olist_master',         df_master),
    ('olist_delivered',      df_delivered),
    ('olist_products_clean', df_products),
    ('olist_sellers_clean',  df_sellers)
]:
    # Count only non-date nulls that matter
    # (master date nulls are intentional)
    total_nulls = df.isnull().sum().sum()
    intentional = 0
    if name == 'olist_master':
        intentional = (
            df['order_approved_at'].isnull().sum() +
            df['order_delivered_carrier_date'].isnull().sum() +
            df['order_delivered_customer_date'].isnull().sum()
        )
    actionable_nulls = total_nulls - intentional
    if actionable_nulls == 0:
        print(f'✅ {name}: clean! '
              f'(intentional date nulls: {intentional})')
    else:
        all_clean = False
        print(f'⚠️  {name}: {actionable_nulls} nulls still need fixing')

print()
if all_clean:
    print('🎉 ALL FILES ARE CLEAN — ready for pgAdmin import!')
    print()
    print('Next steps:')
    print('  1. Re-import olist_master.csv into pgAdmin')
    print('  2. Re-import olist_delivered.csv into pgAdmin')
    print('  3. Run Query 3 and Query 5')
else:
    print('⚠️  Some nulls remain — check output above')

FIXING olist_delivered NULLS...
✅ order_approved_at          → 14 nulls filled with median date
✅ order_delivered_carrier_date → 2 nulls filled with median date
✅ order_delivered_customer_date → 8 nulls filled with median date
✅ payment_type / installments / value → 1 null filled

✅ olist_delivered.csv re-saved!

FINAL VERIFICATION — all 4 cleaned files
✅ olist_master: clean! (intentional date nulls: 4908)
✅ olist_delivered: clean! (intentional date nulls: 0)
⚠️  olist_products_clean: 3058 nulls still need fixing
✅ olist_sellers_clean: clean! (intentional date nulls: 0)

⚠️  Some nulls remain — check output above


In [38]:
df_delivered.isnull().sum()

order_id                         0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
order_year                       0
order_month                      0
order_month_name                 0
order_day_of_week                0
order_hour                       0
delivery_delay_days              0
actual_delivery_days             0
is_late                          0
customer_unique_id               0
customer_city                    0
customer_state                   0
total_items                      0
total_price                      0
total_freight                    0
total_revenue                    0
product_category                 0
payment_type                     0
payment_installments             0
payment_value                    0
review_score                     0
dtype: int64

In [39]:
# ── NULL VALUE CHECK — olist_products_clean ───────────────
print('=' * 50)
print('NULL CHECK — olist_products_clean')
print('=' * 50)
products_nulls = df_products.isnull().sum()
products_nulls = products_nulls[products_nulls > 0]
if len(products_nulls) == 0:
    print('✅ No nulls found!')
else:
    for col, count in products_nulls.items():
        pct = count / len(df_products) * 100
        print(f'   ⚠️  {col}: {count:,} nulls ({pct:.2f}%)')

NULL CHECK — olist_products_clean
   ⚠️  product_category_name: 610 nulls (1.85%)
   ⚠️  product_name_lenght: 610 nulls (1.85%)
   ⚠️  product_description_lenght: 610 nulls (1.85%)
   ⚠️  product_photos_qty: 610 nulls (1.85%)
   ⚠️  product_weight_g: 2 nulls (0.01%)
   ⚠️  product_length_cm: 2 nulls (0.01%)
   ⚠️  product_height_cm: 2 nulls (0.01%)
   ⚠️  product_width_cm: 2 nulls (0.01%)
   ⚠️  product_category_name_english: 610 nulls (1.85%)


In [40]:
# ── FIX NULLS in olist_products_clean ─────────────────────
print('=' * 50)
print('FIXING olist_products_clean NULLS...')
print('=' * 50)

# Fix 1 — 610 products with no category info at all
# These products were sold but seller never filled details
# Fill category with 'unknown_category'
# Fill numeric fields with 0 (no data available)
df_products['product_category_name'] = (
    df_products['product_category_name'].fillna('unknown_category')
)
df_products['product_category_name_english'] = (
    df_products['product_category_name_english'].fillna('unknown_category')
)
df_products['product_name_lenght'] = (
    df_products['product_name_lenght'].fillna(0)
)
df_products['product_description_lenght'] = (
    df_products['product_description_lenght'].fillna(0)
)
df_products['product_photos_qty'] = (
    df_products['product_photos_qty'].fillna(0)
)
print('✅ 610 products with no details:')
print('   product_category_name         → filled with unknown_category')
print('   product_category_name_english → filled with unknown_category')
print('   product_name_lenght           → filled with 0')
print('   product_description_lenght    → filled with 0')
print('   product_photos_qty            → filled with 0')

print()

# Fix 2 — 2 products with no dimension/weight data
# Fill with median values from the rest of the products
# Median is safer than mean for physical measurements
median_weight = df_products['product_weight_g'].median()
median_length = df_products['product_length_cm'].median()
median_height = df_products['product_height_cm'].median()
median_width  = df_products['product_width_cm'].median()

df_products['product_weight_g']  = (
    df_products['product_weight_g'].fillna(median_weight)
)
df_products['product_length_cm'] = (
    df_products['product_length_cm'].fillna(median_length)
)
df_products['product_height_cm'] = (
    df_products['product_height_cm'].fillna(median_height)
)
df_products['product_width_cm']  = (
    df_products['product_width_cm'].fillna(median_width)
)
print('✅ 2 products with no dimensions:')
print(f'   product_weight_g  → filled with median ({median_weight:.0f}g)')
print(f'   product_length_cm → filled with median ({median_length:.0f}cm)')
print(f'   product_height_cm → filled with median ({median_height:.0f}cm)')
print(f'   product_width_cm  → filled with median ({median_width:.0f}cm)')

print()

# Re-save after fix
df_products.to_csv('../cleaned_data/olist_products_clean.csv', index=False)
print('✅ olist_products_clean.csv re-saved!')

print()

# Verify — zero nulls remaining
remaining = df_products.isnull().sum()
remaining = remaining[remaining > 0]
if len(remaining) == 0:
    print('✅ olist_products_clean: ZERO nulls remaining — perfectly clean!')
else:
    print('⚠️  Nulls still remaining:')
    for col, count in remaining.items():
        print(f'   {col}: {count:,}')

FIXING olist_products_clean NULLS...
✅ 610 products with no details:
   product_category_name         → filled with unknown_category
   product_category_name_english → filled with unknown_category
   product_name_lenght           → filled with 0
   product_description_lenght    → filled with 0
   product_photos_qty            → filled with 0

✅ 2 products with no dimensions:
   product_weight_g  → filled with median (700g)
   product_length_cm → filled with median (25cm)
   product_height_cm → filled with median (13cm)
   product_width_cm  → filled with median (20cm)

✅ olist_products_clean.csv re-saved!

✅ olist_products_clean: ZERO nulls remaining — perfectly clean!


In [41]:
df_products.isnull().sum()

product_id                       0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64

In [42]:
# ── NULL VALUE CHECK — olist_sellers_clean ────────────────
print('=' * 50)
print('NULL CHECK — olist_sellers_clean')
print('=' * 50)
sellers_nulls = df_sellers.isnull().sum()
sellers_nulls = sellers_nulls[sellers_nulls > 0]
if len(sellers_nulls) == 0:
    print('✅ No nulls found!')
else:
    for col, count in sellers_nulls.items():
        pct = count / len(df_sellers) * 100
        print(f'   ⚠️  {col}: {count:,} nulls ({pct:.2f}%)')

NULL CHECK — olist_sellers_clean
✅ No nulls found!


In [43]:
df_sellers.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

---
## ✅ Phase 1 Complete!

**What we did:**
- Loaded all 9 CSV files and understood the schema
- Fixed date columns and calculated delivery delay
- Translated product categories to English
- Built a master dataframe by joining all tables
- Saved cleaned data for Phase 2 and 3

**Key early findings:**
- Dataset has ~100k orders from 2016–2018
- Majority of orders are delivered
- Delivery delay and review scores will be key areas to investigate

**Next:** Phase 2 — SQL Analysis (10+ business queries)